# 01 — Exploração dos dados diários

Inspecione a série diária de Bitcoin antes de construir variáveis para os modelos. O notebook espera `data/btc_diario.csv` na raiz do repositório; o arquivo não é versionado.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / 'src').exists() and (RAIZ.parent / 'src').exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.features import carregar_diario

CAMINHO_DADOS = RAIZ / 'data' / 'btc_diario.csv'
diario = carregar_diario(CAMINHO_DADOS)
diario.head()

## Formato, período e qualidade

Verifique se há uma linha por dia, se o índice está ordenado e se os preços e volumes são plausíveis. A função de carga já interrompe a execução quando encontra datas ou valores inválidos.

In [ ]:
print(f'Linhas: {len(diario):,}')
print(f'Período: {diario.index.min().date()} a {diario.index.max().date()}')
print(f'Datas duplicadas: {diario.index.has_duplicates}')
print(f'Dias ausentes no calendário: {len(pd.date_range(diario.index.min(), diario.index.max(), freq="D")) - len(diario)}')
display(diario.dtypes.to_frame('tipo'))
display(diario.describe().T)

## Evolução do preço e do volume

Os gráficos ajudam a reconhecer tendências, mudanças de escala e períodos atípicos. Eles são descritivos e não devem ser usados para escolher um modelo olhando o conjunto de teste.

In [ ]:
fig, eixos = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
diario['close'].plot(ax=eixos[0], title='Fechamento diário BTC/USD')
eixos[0].set_ylabel('USD')
diario['volume'].plot(ax=eixos[1], color='darkorange', title='Volume diário')
eixos[1].set_ylabel('Volume')
plt.tight_layout()
plt.show()

In [ ]:
retorno = diario['close'].pct_change(fill_method=None)
display(retorno.describe().to_frame('retorno diário'))
retorno.plot(figsize=(14, 4), title='Retorno percentual diário do fechamento')
plt.ylabel('Retorno')
plt.show()

## Registro das observações

Antes de seguir, anote no PR o período analisado, falhas encontradas, limitações da fonte e qualquer decisão tomada para gerar o arquivo diário. Não remova dias silenciosamente: investigue lacunas na série de origem.